In [8]:
using MarineHydro
using DifferentiationInterface 
import ForwardDiff 



In [9]:

# Hydrostatics
function hydrostatic_program(x)
    r1 = x[1]
    r2 = x[2]
    d1 = x[3]
    d2 = x[4]
    rho_w = 1025.0 # density of fluid [kg/m^3]
    g = 9.81 # acceleration due to gravity [m/s^2]
    K = rho_w * g * pi * r1^2 # hydrostatic stiffness [kg/s^2]
    d3 = (r2/(r1-r2))*d2
    dt = d2+d3
    Vt = pi * r1^2 * d1 + pi * r1^2 * dt/3 - pi * r2^2 * d3/3
    M = rho_w * Vt # mass of body [kg]
    # display("Mass: $M")
    return [M, K]
end

# Radiation
function radiation_program(mesh, omega, dof) 
    A, B = calculate_radiation_forces(mesh,dof,omega)
    # display("Damping: $B")
    return [A, B]
end

# Diffraction
function diffraction_program(mesh, omega, dof) 
    F_D = DiffractionForce(mesh,omega,dof)
    return [real(F_D),imag(F_D)]
end

function incident_program(mesh, omega, dof) 
    F_FK = FroudeKrylovForce(mesh,omega,dof)
    return [real(F_FK),imag(F_FK)]
end

# Everything
dof = [0.0,0.0,1.0]

function compute_for_omega(mesh, omega)
    # r1 = x[1]
    # r2 = x[2]
    # d1 = x[3]
    # d2 = x[4]

    # M, K = hydrostatic_program(r1, r2, d1, d2)
    A, B = radiation_program(mesh, omega, dof)
    F_D_real, F_D_imag = diffraction_program(mesh, omega, dof) 
    F_FK_real, F_FK_imag = incident_program(mesh, omega, dof)
    F_ex_real = F_D_real + F_FK_real
    F_ex_imag = F_D_imag + F_FK_imag
    # add negatives to imag since wot uses +i omega t instead of -i omega t
    return [A, B, F_D_real, -F_D_imag, F_FK_real, -F_FK_imag, F_ex_real, -F_ex_imag]
end

# function compute_for_omega(x, omega)
#     # set_rho!(1025.0)
#     mesh = wavebot_mesh(x[1],x[2],x[3],x[4],(3,5,3),20)
#     val = all_programs(mesh, x, omega)
#     return val
# end

backend = AutoForwardDiff()

function compute_vals(x, omegas)
    set_rho!(1025.0)
    mesh = wavebot_mesh(x[1],x[2],x[3],x[4],(3,3,3),10)
    M_val, K_val = hydrostatic_program(x)
    results_per_omega = [compute_for_omega(mesh, omega) for omega in omegas]
    vec_reduced = reduce(vcat, results_per_omega)
    
    return [M_val; K_val; vec_reduced]
end

function compute_all(x_val, omegas)
    return value_and_jacobian(x -> compute_vals(x, omegas), backend, x_val)
end






    # AD_grads = value_and_jacobian(x-> hydrostatic_program(x), backend, x_val)


    # AD_grads = [value_and_jacobian(x -> compute_for_omega(x, omega), backend, x_val) for omega in omegas]
    

compute_all (generic function with 1 method)

In [10]:
x_val = [1, 0.1, 0.1, 1]
omegas = [0.2, 0.3]
n = 5
AD_grads = compute_all(x_val, omegas)
# compute_for_omega(x_val, n*2*pi*0.3)
# AD_grads = compute_all(x_val, omegas)
# display(AD_grads)

([1513.4622608668826, 31589.49953000877, 1707.893775298319, 3.57931125491301, -68.30856967439422, 0.715862264420442, 29495.01765071831, -1.3335686721571705e-15, 29426.709081043915, 0.7158622644204407, 1726.6201642401484, 11.950501479065059, -155.35895195739087, 3.585150717569688, 29424.277042376398, -4.466912950640278e-16, 29268.918090419007, 3.5851507175696877], [2898.1192229365843 1288.0529879718154 3220.132469929538 1191.4490138739288; 63178.99906001754 0.0 0.0 0.0; … ; 58333.299117629445 -54.01993917893249 -229.86309801294476 -52.279355849943656; 14.289889914587395 -0.012632749532043934 -0.05832116389516626 -0.01231289841539065])

In [11]:
#arbitrary axisymmetric geom


# Hydrostatics
function hydrostatic_program(x)
    r_side = x[1:end-1]
    draft = x[end]
    n_side = length(r_side)
    
    z_side = range(-1e-3, -draft, length=n_side)

    rho_w = 1025.0 
    g = 9.81 
    K = rho_w * g * pi * r_side[1]^2 


    volume = 0.0
    for i in 1:(n_side-1)
        h_mid = abs(z_side[i] - z_side[i+1])
        r1 = r_side[i]
        r2 = r_side[i+1]

        volume += (pi * h_mid / 3) * (r1^2 + r1 * r2 + r2^2)
    end
    
    M = rho_w * volume 

    return [M, K]
end

# Radiation
function radiation_program(mesh, omega, dof) 
    A, B = calculate_radiation_forces(mesh,dof,omega)
    # display("Damping: $B")
    return [A, B]
end

# Diffraction
function diffraction_program(mesh, omega, dof) 
    F_D = DiffractionForce(mesh,omega,dof)
    return [real(F_D),imag(F_D)]
end

function incident_program(mesh, omega, dof) 
    F_FK = FroudeKrylovForce(mesh,omega,dof)
    return [real(F_FK),imag(F_FK)]
end

# Everything
dof = [0.0,0.0,1.0]

function compute_for_omega(mesh, omega)

    # M, K = hydrostatic_program(r1, r2, d1, d2)
    A, B = radiation_program(mesh, omega, dof)
    F_D_real, F_D_imag = diffraction_program(mesh, omega, dof) 
    F_FK_real, F_FK_imag = incident_program(mesh, omega, dof)
    # add negatives to imag since wot uses +i omega t instead of -i omega t
    return [A, B, F_D_real, -F_D_imag, F_FK_real, -F_FK_imag]
end

backend = AutoForwardDiff()


function arbitrary_axisymmetric_mesh(x::Vector, show_plot)

    r_side = x[1:end-1]
    draft = x[end]
    n_side = length(r_side)

    z_side = collect(range(-1e-3, -draft, length=n_side))

    panel_z_len = abs(z_side[1]-z_side[2])
    last_z = z_side[end]

    n_bottom = 3

    z_bottom = ones(n_bottom) * (-draft)
    r_bottom = collect(range(r_side[end], 0.0, length=n_bottom+1))[2:end]

    z_vals = vcat(z_side, z_bottom)
    r_vals = vcat(r_side, r_bottom)

    n_theta = 10

    MH_mesh = axisymmetric_mesh(r_vals,z_vals,n_theta,show_plot)

    return MH_mesh
end

arbitrary_axisymmetric_mesh(x::Vector) = arbitrary_axisymmetric_mesh(x, false)


function compute_vals(x, omegas)
    set_rho!(1025.0)
    mesh = arbitrary_axisymmetric_mesh(x)    
    M_val, K_val = hydrostatic_program(x)
    results_per_omega = [compute_for_omega(mesh, omega) for omega in omegas]
    vec_reduced = reduce(vcat, results_per_omega)
    
    return [M_val; K_val; vec_reduced]
end

function compute_all(x_val, omegas)
    return value_and_jacobian(x -> compute_vals(x, omegas), backend, x_val)
end




compute_all (generic function with 1 method)

In [12]:

r_initial = 0.5
n_side = 3 # number of design vars-1 (draft is also a design var)

# inputs
r_side = ones(n_side) * r_initial
# r_side = collect(range(0.5, 0.001, length=n_side)) # last radius cannot be zero
draft = 1.5
x_val = vcat(r_side,draft)
omegas = [0.2, 0.3]
AD_grads = compute_all(x_val, omegas)
display(AD_grads)

([1206.7446431060944, 7897.374882502192, 280.3633392481834, 0.22203817067263548, -11.158168128619572, 0.044408601295532965, 7342.858093094276, 6.938893903907228e-17, 281.21693704594634, 0.7345573544113135, -25.025393249060844, 0.22039401693306981, 7286.920414685993, -2.3592239273284576e-16], [1206.7446431060941 2413.4892862121887 1206.7446431060944 805.0331174823845; 31589.49953000877 0.0 0.0 0.0; … ; 29449.80173278122 -201.8065387888496 -100.38166883900692 -66.85248086867882; -5.329070518200751e-15 1.7763568394002505e-15 5.10702591327572e-15 2.4936649967166602e-18])

In [14]:
AD_grads[1]

14-element Vector{Float64}:
 1206.7446431060944
 7897.374882502192
  280.3633392481834
    0.22203817067263548
  -11.158168128619572
    0.044408601295532965
 7342.858093094276
    6.938893903907228e-17
  281.21693704594634
    0.7345573544113135
  -25.025393249060844
    0.22039401693306981
 7286.920414685993
   -2.3592239273284576e-16